# **Part 2 Solving MDP and Reinforcement Learning Problems Using a Grid World**

## **Grid World Specification**

The environment is a 5 × 5 grid world. Each grid cell corresponds to a state represented by its coordinates ($𝑥$, $𝑦$), where $𝑥$, $𝑦$ $∈$ {$0,1,2,3,4$}.  
-  Start state: $(0, 0)$ (bottom-left corner)  
- Goal state: $(4, 4)$ (top-right corner)  
- Terminal state: The episode ends when the agent reaches the goal state.  
- Roadblocks: The agent cannot enter cells $(2, 1)$ and $(2, 3)$. These cells act as obstacles and
must be avoided when planning or learning a policy.  
At each time step, the agent can take one of four actions: move Up, Down, Left, or Right. If an action
would move the agent outside the grid or into a roadblock, the agent remains in its current state.

Rewards  
- Each action incurs a step cost of −1.  
- Reaching the goal state yields a reward of +10, after which the episode ends.


### **Imports and Constants**

In [4]:
import pandas as pd
import random
from dataclasses import dataclass
from collections import defaultdict

# Grid settings
N = 5
START = (0, 0)
GOAL = (4, 4)
BLOCKS = {(1, 2), (3, 2)}

# Actions and movement
ACTIONS = ["U", "D", "L", "R"]
DELTA = {"U": (0, 1), "D": (0, -1), "L": (-1, 0), "R": (1, 0)}

# MDP settings
STEP_COST = -1
GOAL_REWARD = 10

### **Tabular Representation**

In [5]:
ARROW = {"U": "↑", "D": "↓", "L": "←", "R": "→", None: "·"}

def V_table(V):
    """
    Returns a 5x5 DataFrame indexed by y (top to bottom) and columns x (left to right).
    """
    grid = []
    for y in reversed(range(N)):
        row = []
        for x in range(N):
            s = (x, y)
            if s in BLOCKS:
                row.append(None)
            elif s == GOAL:
                row.append(0.0)
            else:
                row.append(V.get(s, None))
        grid.append(row)
    df = pd.DataFrame(grid, columns=list(range(N)), index=list(reversed(range(N))))
    return df

def policy_table(pi):
    """
    Returns a 5x5 DataFrame of arrow symbols.
    """
    grid = []
    for y in reversed(range(N)):
        row = []
        for x in range(N):
            s = (x, y)
            if s in BLOCKS:
                row.append("####")
            elif s == GOAL:
                row.append("GOAL")
            else:
                row.append(ARROW[pi.get(s, None)])
        grid.append(row)
    df = pd.DataFrame(grid, columns=list(range(N)), index=list(reversed(range(N))))
    return df

def Q_table(Q):
    """
    Returns a DataFrame with index=states and columns=actions.
    """
    rows = []
    for s in STATES:
        if s in BLOCKS:
            continue
        row = {"state": s}
        for a in ACTIONS:
            row[a] = Q.get((s, a), 0.0)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("state").sort_index()
    return df

### **States + Basic Helper**

In [6]:
STATES = [(x, y) for x in range(N) for y in range(N) if (x, y) not in BLOCKS]

def is_terminal(s):
    return s == GOAL

def in_bounds(cell):
    x, y = cell
    return 0 <= x < N and 0 <= y < N

def is_blocked(cell):
    return cell in BLOCKS

## **Task 1**

The environment model is fully known. The transition dynamics are deterministic, and the discount factor is 𝜸 = 𝟎. 𝟗. You are required to implement two planning algorithms:  
- *Value Iteration*:
Compute the optimal value function using the Bellman optimality equation and derive the optimal policy.  
- *Policy Iteration*:
Start from an initial policy and alternate between policy evaluation and policy
improvement until convergence. Report the resulting value functions and policies, and compare the policies obtained from value iteration and policy iteration.

### **Deterministic Transition Model**

The function step(s, a) defines the MDP model for planning:
- If s is terminal, it transitions to itself with reward $0$.
- Otherwise, apply the movement for action $a$.
- If the movement goes outside the grid or into a blocked cell, the agent stays in place.
- Reward is $−1$ for normal steps and $+10$ if the resulting state is the goal.

In [7]:
GAMMA = 0.9

def step_deterministic(s, a):
    """
    Returns (s_next, reward).
    Deterministic movement with obstacles and boundary conditions.
    """
    if is_terminal(s):
        return s, 0

    dx, dy = DELTA[a]
    candidate = (s[0] + dx, s[1] + dy)

    if in_bounds(candidate) and not is_blocked(candidate):
        s_next = candidate
    else:
        s_next = s

    reward = GOAL_REWARD if s_next == GOAL else STEP_COST
    return s_next, reward

#### **1. Value Iteration**

Value iteration computes the optimal value function using the Bellman optimality equation:
$$
V^*(s) = \max_a \left[ R(s,a,s') + \gamma V^*(s') \right]
$$
Because the environment is deterministic, each $(s,a)$ pair produces a single next state $s'$. We repeatedly update $V(s)$ for all non-terminal states until convergence (i.e., the maximum change is below a small tolerance $\theta$).  
After convergence, the optimal policy is derived greedily from $V^*$ by choosing the action with the highest one-step lookahead value.  
We also compute the action-value function explicitly from the final $V$:  
$$
Q(s,a) = R(s,a,s') + \gamma V(s')
$$

In [8]:
@dataclass
class VIResult:
    V: dict
    Q: dict
    pi: dict
    sweeps: int
    backups: int

def value_iteration(theta=1e-10, max_sweeps=10000):
    V = {s: 0.0 for s in STATES}
    sweeps = 0
    backups = 0

    while True:
        sweeps += 1
        delta = 0.0

        for s in STATES:
            if is_terminal(s):
                V[s] = 0.0
                continue

            best = float("-inf")
            for a in ACTIONS:
                s_next, r = step_deterministic(s, a)
                backups += 1
                q = r + GAMMA * V[s_next]
                if q > best:
                    best = q

            delta = max(delta, abs(best - V[s]))
            V[s] = best

        if delta < theta or sweeps >= max_sweeps:
            break

    # Build Q and greedy policy from converged V
    Q = {}
    pi = {}
    for s in STATES:
        if is_terminal(s):
            pi[s] = None
            for a in ACTIONS:
                Q[(s, a)] = 0.0
            continue

        best = float("-inf")
        besta = None
        for a in ACTIONS:
            s_next, r = step_deterministic(s, a)
            q = r + GAMMA * V[s_next]
            Q[(s, a)] = q
            if q > best:
                best = q
                besta = a
        pi[s] = besta

    return VIResult(V, Q, pi, sweeps, backups)

#### **2. Policy Iteration**

Policy iteration alternates between two steps:
- Policy Evaluation:
Compute $V^\pi(s)$ for the current policy $\pi$ until convergence.
$$
V^\pi(s) = R(s,\pi(s),s') + \gamma V^\pi(s')
$$
- Policy Improvement
Update the policy to be greedy with respect to $V^\pi$:
$$
\pi_{\text{new}}(s) = \arg\max_a \left[ R(s,a,s') + \gamma V^\pi(s') \right]
$$
The algorithm stops when the policy no longer changes (policy stability).  
We then compute the action-value function $Q(s,a)$ from the final value function in the same tabular manner as in value iteration:  
$$
Q(s,a) = R(s,a,s') + \gamma V(s')
$$

#### **Policy Evaluation and Improvement**

In [9]:
@dataclass
class PIResult:
    V: dict
    Q: dict
    pi: dict
    improv_steps: int
    eval_updates: int

def policy_evaluation(pi, theta=1e-10):
    V = {s: 0.0 for s in STATES}
    updates = 0

    while True:
        delta = 0.0
        for s in STATES:
            if is_terminal(s):
                V[s] = 0.0
                continue

            a = pi[s]
            s_next, r = step_deterministic(s, a)
            v_new = r + GAMMA * V[s_next]
            updates += 1
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new

        if delta < theta:
            break

    return V, updates

def policy_improvement(V, pi):
    stable = True
    for s in STATES:
        if is_terminal(s):
            continue

        old = pi[s]
        best = float("-inf")
        besta = None

        for a in ACTIONS:
            s_next, r = step_deterministic(s, a)
            q = r + GAMMA * V[s_next]
            if q > best:
                best = q
                besta = a

        pi[s] = besta
        if besta != old:
            stable = False

    return stable, pi

def policy_iteration():
    # Initial policy (arbitrary): always Up unless terminal
    pi = {s: ("U" if not is_terminal(s) else None) for s in STATES}

    total_eval_updates = 0
    improv_steps = 0

    while True:
        V, updates = policy_evaluation(pi)
        total_eval_updates += updates

        stable, pi = policy_improvement(V, pi)
        improv_steps += 1

        if stable:
            break

    # Build Q from final V
    Q = {}
    for s in STATES:
        for a in ACTIONS:
            if is_terminal(s):
                Q[(s, a)] = 0.0
            else:
                s_next, r = step_deterministic(s, a)
                Q[(s, a)] = r + GAMMA * V[s_next]

    return PIResult(V, Q, pi, improv_steps, total_eval_updates)

### **Comparision between two algorithms**

#### **Value Iteration**

In [10]:
vi = value_iteration()

display(V_table(vi.V).round(2))
display(Q_table(vi.Q).round(2))
display(policy_table(vi.pi))

,0,1,2,3,4
4,4.58,6.20,8.00,10.00,0.00
3,3.12,4.58,6.20,8.00,10.00
2,1.81,NaN,4.58,NaN,8.00
1,0.63,1.81,3.12,4.58,6.20
0,-0.43,0.63,1.81,3.12,4.58


,U,D,L,R
state,,,,
"(0, 0)",-0.43,-1.39,-1.39,-0.43
"(0, 1)",0.63,-1.39,-0.43,0.63
"(0, 2)",1.81,-0.43,0.63,0.63
"(0, 3)",3.12,0.63,1.81,3.12
"(0, 4)",3.12,1.81,3.12,4.58
"(1, 0)",0.63,-0.43,-1.39,0.63
"(1, 1)",0.63,-0.43,-0.43,1.81
"(1, 3)",4.58,3.12,1.81,4.58
"(1, 4)",4.58,3.12,3.12,6.20


,0,1,2,3,4
4,→,→,→,→,GOAL
3,↑,↑,↑,↑,↑
2,↑,####,↑,####,↑
1,↑,→,↑,→,↑
0,↑,↑,↑,↑,↑


#### **Policy Iteration**

In [11]:
pi = policy_iteration()

# Policy Iteration outputs
display(V_table(pi.V).round(2))
display(Q_table(pi.Q).round(2))
display(policy_table(pi.pi))

,0,1,2,3,4
4,4.58,6.20,8.00,10.00,0.00
3,3.12,4.58,6.20,8.00,10.00
2,1.81,NaN,4.58,NaN,8.00
1,0.63,1.81,3.12,4.58,6.20
0,-0.43,0.63,1.81,3.12,4.58


,U,D,L,R
state,,,,
"(0, 0)",-0.43,-1.39,-1.39,-0.43
"(0, 1)",0.63,-1.39,-0.43,0.63
"(0, 2)",1.81,-0.43,0.63,0.63
"(0, 3)",3.12,0.63,1.81,3.12
"(0, 4)",3.12,1.81,3.12,4.58
"(1, 0)",0.63,-0.43,-1.39,0.63
"(1, 1)",0.63,-0.43,-0.43,1.81
"(1, 3)",4.58,3.12,1.81,4.58
"(1, 4)",4.58,3.12,3.12,6.20


,0,1,2,3,4
4,→,→,→,→,GOAL
3,↑,↑,↑,↑,↑
2,↑,####,↑,####,↑
1,↑,→,↑,→,↑
0,↑,↑,↑,↑,↑


#### **Compare**

In [12]:
same = all(vi.pi[s] == pi.pi[s] for s in STATES)
print("Policies identical?", same)
print("Value Iteration sweeps:", vi.sweeps, " | Bellman backups:", vi.backups)
print("Policy Iteration improvement steps:", pi.improv_steps, " | Eval updates:", pi.eval_updates)

Policies identical? True
Value Iteration sweeps: 9  | Bellman backups: 792
Policy Iteration improvement steps: 6  | Eval updates: 19756


## **Task 2**

If the agent does not have access to the transition model, you are required to implement Monte
Carlo (MC) prediction and control using complete episodes.  
Specifically, you are required to use an εgreedy strategy for policy improvement. Use a fixed $ε$ value of $0.1$ throughout training. Estimate state – action values using sampled returns. Train the agent over multiple episodes starting from the start state.  
After training, extract the learned policy and compare it with the optimal policies from Task 1.

### **Monte Carlo Theory**

For a trajectory, the return is defined as:
$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots
$$
Monte Carlo methods estimate the action-value function as:
$$
Q(s,a) = \mathbb{E}\left[ G_t \mid S_t = s, A_t = a \right]
$$
Since the transition model is unknown, this expectation is approximated by averaging sampled returns collected across multiple episodes.

We use *First-Visit MC*:

- Update $Q(s,a)$ only the first time $(s,a)$ appears in an episode.
- Maintain a list of returns for each $(s,a)$ pair.
- Update the estimate using the empirical mean of observed returns.

### **$ε$-Greedy Policy**

To balance exploration and exploitation:
- Each non-greedy action receives probability:
$$
\frac{\varepsilon}{|A|}
$$
- The greedy action receives probability:
$$
1 - \varepsilon + \frac{\varepsilon}{|A|}
$$
This guarantees a valid probability distribution (i.e., all probabilities sum to 1).

We keep $\varepsilon$ fixed at 0.1 throughout training.

#### **Stochastic Environment Sampler**

In [13]:
EPS = 0.1  # fixed epsilon for ε-greedy, as required

# Perpendicular directions for each intended action
PERP = {
    "U": ("L", "R"),
    "D": ("L", "R"),
    "L": ("U", "D"),
    "R": ("U", "D"),
}

def valid_next(s, a):
    """Deterministic move for action a with boundary/roadblock handling."""
    if is_terminal(s):
        return s
    dx, dy = DELTA[a]
    cand = (s[0] + dx, s[1] + dy)
    if in_bounds(cand) and not is_blocked(cand):
        return cand
    return s

def step_stochastic_sample(s, intended_a):
    """
    Sample one transition from the stochastic environment:
    - 0.8: intended action
    - 0.1: perpendicular left
    - 0.1: perpendicular right
    Reward:
    - -1 per step
    - +10 on entering GOAL (terminal)
    """
    if is_terminal(s):
        return s, 0, True

    r = random.random()
    if r < 0.8:
        executed_a = intended_a
    elif r < 0.9:
        executed_a = PERP[intended_a][0]
    else:
        executed_a = PERP[intended_a][1]

    s_next = valid_next(s, executed_a)
    reward = GOAL_REWARD if s_next == GOAL else STEP_COST
    done = (s_next == GOAL)
    return s_next, reward, done

#### **$ε$-greedy Action Selection**

In [14]:
def epsilon_greedy_probs(Q, s, eps=EPS):
    nA = len(ACTIONS)
    qs = {a: Q[(s, a)] for a in ACTIONS}
    max_q = max(qs.values())
    greedy = [a for a, v in qs.items() if v == max_q]

    probs = {a: eps / nA for a in ACTIONS}
    greedy_mass = 1.0 - (nA - 1) * (eps / nA)

    for a in greedy:
        probs[a] += greedy_mass / len(greedy)

    return probs

def sample_from_probs(probs):
    r = random.random()
    cumulative = 0.0
    for a, p in probs.items():
        cumulative += p
        if r <= cumulative:
            return a
    return a

def select_action_epsilon_greedy(Q, s):
    probs = epsilon_greedy_probs(Q, s, EPS)
    return sample_from_probs(probs)

### **Monte Carlo Control Algorithm**

For each episode:

1. Generate a complete trajectory using ε-greedy policy.
2. Compute returns $G_t$ backward.
3. For each first occurrence of $(s,a)$:
   - Append return to `Returns[(s,a)]`
   - Update $Q(s,a)$ as empirical average.

Policy improvement is implicit because future action selection uses updated $Q$ via $ε$-greedy.

#### **MC Control Implementation**

In [15]:
def mc_control_first_visit(num_episodes=30000, max_steps=200, seed=0):
    random.seed(seed)

    Q = defaultdict(float)
    Returns = defaultdict(list)

    for ep in range(num_episodes):
        episode = []
        s = START

        # Generate complete episode
        for t in range(max_steps):
            a = select_action_epsilon_greedy(Q, s)
            s_next, r, done = step_stochastic_sample(s, a)
            episode.append((s, a, r))
            s = s_next
            if done:
                break

        # Compute returns
        G = 0.0
        returns_t = [0.0] * len(episode)
        for i in reversed(range(len(episode))):
            _, _, r = episode[i]
            G = r + GAMMA * G
            returns_t[i] = G

        # First-visit updates
        visited = set()
        for i, (s, a, _) in enumerate(episode):
            if (s, a) in visited:
                continue
            visited.add((s, a))

            Returns[(s, a)].append(returns_t[i])
            Q[(s, a)] = sum(Returns[(s, a)]) / len(Returns[(s, a)])

    return Q

#### **Extract V and Policy**

In [16]:
def greedy_policy_from_Q(Q):
    pi = {}
    for s in STATES:
        if is_terminal(s):
            pi[s] = None
        else:
            pi[s] = max(ACTIONS, key=lambda a: Q[(s, a)])
    return pi

def V_from_Q(Q):
    V = {}
    for s in STATES:
        if is_terminal(s):
            V[s] = 0.0
        else:
            V[s] = max(Q[(s, a)] for a in ACTIONS)
    return V

### **Training and Result**

We train the agent for 50,000 episodes starting from (0,0).

In [18]:
Q_mc = mc_control_first_visit(num_episodes=50000)

pi_mc = greedy_policy_from_Q(Q_mc)
V_mc = V_from_Q(Q_mc)

display(V_table(V_mc).round(2))
display(Q_table(Q_mc).round(2))
display(policy_table(pi_mc))

,0,1,2,3,4
4,-2.75,0.79,2.66,4.74,0.00
3,-2.08,-0.68,1.27,6.41,9.15
2,-3.14,NaN,-0.25,NaN,6.62
1,-3.92,-2.88,-1.58,-0.86,3.88
0,-4.61,-3.71,-1.30,0.15,1.80


,U,D,L,R
state,,,,
"(0, 0)",-4.61,-5.05,-5.14,-9.92
"(0, 1)",-3.92,-4.91,-4.66,-10.00
"(0, 2)",-3.14,-4.43,-3.83,-9.35
"(0, 3)",-3.29,-3.53,-3.12,-2.08
"(0, 4)",-3.16,-2.75,-3.45,-10.00
"(1, 0)",-3.71,-4.21,-4.93,-9.90
"(1, 1)",-3.68,-4.16,-4.26,-2.88
"(1, 3)",-0.68,-1.70,-2.59,-4.79
"(1, 4)",-0.64,-1.52,-2.98,0.79


,0,1,2,3,4
4,↓,→,→,↓,GOAL
3,→,↑,↑,→,↑
2,↑,####,↑,####,↑
1,↑,→,↑,↓,↑
0,↑,↑,→,→,↑


### **Comparision with Task 1**

In [16]:
def policy_match_ratio(pi_learned, pi_ref):
    same = 0
    total = 0
    for s in STATES:
        if is_terminal(s):
            continue
        total += 1
        if pi_learned.get(s) == pi_ref.get(s):
            same += 1
    return same, total, same / total

same_vi, total, ratio_vi = policy_match_ratio(pi_mc, vi.pi)
same_pi, total, ratio_pi = policy_match_ratio(pi_mc, pi.pi)

print(f"Match vs Value Iteration: {same_vi}/{total} = {ratio_vi:.2%}")
print(f"Match vs Policy Iteration: {same_pi}/{total} = {ratio_pi:.2%}")

Match vs Value Iteration: 17/22 = 77.27%
Match vs Policy Iteration: 17/22 = 77.27%
